# Lesson 6.4: What Is a Reranker and When Do You Actually Need One?

**Companion notebook for Lesson 6.4**

---

| Section | What you will build |
|---|---|
| 1. Two-Stage Pattern | Pipeline diagram + latency intuition |
| 2. Bi-Encoder vs Cross-Encoder | Side-by-side scoring on the OOM example |
| 3. Inside a Cross-Encoder | Tokenisation format, attention, step-by-step scoring |
| 4. Why CE Can't Do Retrieval | Concrete latency math |
| 5. Latency Analysis | Measure reranking time vs candidate set size |
| 6. The File Upload Example | Full hybrid → rerank pipeline; position 7 → position 1 |
| 7. Retrieval Diagnostic | `diagnose_retrieval()` — is your problem Stage 1 or Stage 2? |
| 8. Complete RAG Pipeline | Production-ready two-stage retrieval function |
| 9. Cohere API | Managed reranking in 5 lines |

**Required:** `sentence-transformers`, `rank-bm25`, `numpy`, `matplotlib`  
**Optional (tokenisation viz):** `transformers` (already installed with sentence-transformers)  
**Optional (Section 9):** `cohere`

In [ ]:
# Uncomment to install
# !pip install sentence-transformers rank-bm25 numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import time
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Setup complete.')

---
## 1. The Two-Stage Retrieval Pattern

The blog's core tension:

> **The best retrieval method is too slow to run on your entire corpus.  
> The fastest retrieval method isn't accurate enough for your final answer.**

The solution: retrieve-then-rerank.

In [ ]:
# ASCII pipeline diagram
pipeline = '''
┌─────────────────────────────────────────────────────────────────────────┐
│                    TWO-STAGE RETRIEVAL PIPELINE                         │
└─────────────────────────────────────────────────────────────────────────┘

  User Query
      │
      ▼
┌─────────────┐    Fast (10-50 ms)    ┌─────────────────────────────────┐
│  STAGE 1    │ ──────────────────▶   │  Top-20 candidates              │
│  Retrieval  │   BM25 + Semantic +   │  (broad net, high recall)       │
│  (Recall)   │   RRF fusion          │                                 │
└─────────────┘                       └──────────────┬──────────────────┘
                                                     │
                                                     ▼
                              Slow but accurate (100-300 ms)
                              ┌──────────────────────────────┐
                              │  STAGE 2  Reranking          │
                              │  Cross-encoder reads         │
                              │  query + doc TOGETHER        │
                              │  (high precision)            │
                              └──────────────┬───────────────┘
                                             │
                                             ▼
                              ┌──────────────────────────────┐
                              │  Top-3 for LLM               │
                              │  (only the very best)        │
                              └──────────────────────────────┘

  Analogy:
  Stage 1 = Resume screen   (fast, filters 1000 → 20)
  Stage 2 = Deep interview  (slow, evaluates each candidate carefully)
  Stage 3 = Hiring decision (LLM works with whoever made the cut)
'''
print(pipeline)

---
## 2. Bi-Encoder vs Cross-Encoder

| Property | Bi-Encoder (Stage 1) | Cross-Encoder (Stage 2) |
|---|---|---|
| How it encodes | Query and doc **independently** | Query + doc **together** in one pass |
| Speed | Fast (pre-computed doc vectors) | Slow (must run per query-doc pair) |
| Interaction | Only cosine similarity of vectors | Full cross-attention between all tokens |
| Scales to 1M docs? | Yes | No (minutes per query) |

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder, util

print('Loading bi-encoder (all-MiniLM-L6-v2)...')
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')

print('Loading cross-encoder reranker (ms-marco-MiniLM-L-6-v2)...')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print('Both models ready.')

In [ ]:
# The OOM example from the blog
query = 'How do I fix OOM errors in Java?'

docs = [
    'To resolve memory errors in Java, increase the heap size using the -Xmx flag when starting the JVM.',
    'Java garbage collection overview: understanding generational GC and tuning strategies for production.',
    'OutOfMemoryError occurs when the JVM cannot allocate an object because it is out of memory and garbage collection cannot free up more memory.',
    'Common Java exceptions and how to handle them: NullPointerException, ArrayIndexOutOfBoundsException, ClassCastException.',
    'Memory management best practices for production Java systems: monitoring heap usage, profiling, and GC tuning.',
]

labels = [
    'Fix: use -Xmx flag',
    'GC overview & tuning',
    'OutOfMemoryError definition',    # <- most directly relevant
    'Common Java exceptions',
    'Memory mgmt best practices',
]

# ── Bi-encoder: encode independently ──────────────────────────────────────────
query_emb = bi_encoder.encode(query, convert_to_tensor=True)
doc_embs  = bi_encoder.encode(docs,  convert_to_tensor=True)
bi_scores = util.cos_sim(query_emb, doc_embs)[0].numpy()
bi_order  = np.argsort(bi_scores)[::-1]

print('Bi-Encoder scores (cosine similarity):')
for rank, idx in enumerate(bi_order, 1):
    print(f'  #{rank}: {labels[idx]:<35}  score = {bi_scores[idx]:.4f}')

print()

# ── Cross-encoder: encode together ────────────────────────────────────────────
pairs       = [[query, doc] for doc in docs]
ce_scores   = cross_encoder.predict(pairs)
ce_order    = np.argsort(ce_scores)[::-1]

print('Cross-Encoder scores (relevance logit):')
for rank, idx in enumerate(ce_order, 1):
    print(f'  #{rank}: {labels[idx]:<35}  score = {ce_scores[idx]:.4f}')

In [ ]:
# Side-by-side ranking comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

best_doc_idx = 2  # 'OutOfMemoryError definition' is ground-truth best

def bar_color(idx):
    return '#2E7D32' if idx == best_doc_idx else '#90CAF9'

y = np.arange(len(docs))

# Left: bi-encoder
bi_colors = [bar_color(idx) for idx in bi_order]
axes[0].barh(y, [bi_scores[idx] for idx in bi_order], color=bi_colors, alpha=0.85)
axes[0].set_yticks(y)
axes[0].set_yticklabels([f'#{r+1}: {labels[idx]}' for r, idx in enumerate(bi_order)], fontsize=9)
axes[0].set_xlabel('Cosine Similarity')
axes[0].set_title('Bi-Encoder Ranking\n(query and doc encoded SEPARATELY)', fontweight='bold')

# Right: cross-encoder (normalise for display)
ce_min, ce_max = ce_scores.min(), ce_scores.max()
ce_norm = (ce_scores - ce_min) / (ce_max - ce_min)
ce_colors = [bar_color(idx) for idx in ce_order]
axes[1].barh(y, [ce_norm[idx] for idx in ce_order], color=ce_colors, alpha=0.85)
axes[1].set_yticks(y)
axes[1].set_yticklabels([f'#{r+1}: {labels[idx]}' for r, idx in enumerate(ce_order)], fontsize=9)
axes[1].set_xlabel('Relevance Score (normalised)')
axes[1].set_title('Cross-Encoder Ranking\n(query and doc read TOGETHER)', fontweight='bold')

legend_handles = [
    mpatches.Patch(color='#2E7D32', label='Ground-truth best document'),
    mpatches.Patch(color='#90CAF9', label='Other documents'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2,
           fontsize=10, bbox_to_anchor=(0.5, -0.06))
fig.suptitle(f'Query: {query!r}', style='italic', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

bi_pos_of_best  = list(bi_order).index(best_doc_idx) + 1
ce_pos_of_best  = list(ce_order).index(best_doc_idx) + 1
print(f'Best document at bi-encoder rank #{bi_pos_of_best}')
print(f'Best document at cross-encoder rank #{ce_pos_of_best}')
print()
print('The cross-encoder recognises that "OutOfMemoryError" = "OOM errors" and')
print('"JVM cannot allocate" = the root cause the user is trying to fix.')
print('The bi-encoder\'s cosine similarity misses this because embeddings are')
print('averaged over the full text, diluting the OOM signal.')

---
## 3. Inside a Cross-Encoder (monoBERT Style)

The cross-encoder processes each query-document pair as a **single input sequence**:

```
[CLS] query tokens [SEP] document tokens [SEP]
```

Every query token can **attend** to every document token through all Transformer layers.
The `[CLS]` token's final representation summarises the query-document relationship,
and a linear head converts it to a relevance score.

In [ ]:
# Visualise the [CLS] q [SEP] d [SEP] input format
try:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained('cross-encoder/ms-marco-MiniLM-L-6-v2')

    q_short = 'How do I fix OOM errors?'
    d_short = 'OutOfMemoryError occurs when JVM runs out of memory.'

    encoded = tokenizer(q_short, d_short, truncation=True, max_length=32)
    tokens  = tokenizer.convert_ids_to_tokens(encoded['input_ids'])
    type_ids = encoded.get('token_type_ids', [0] * len(tokens))

    # Assign roles
    roles = []
    in_doc = False
    sep_count = 0
    for tok, seg in zip(tokens, type_ids):
        if tok == '[CLS]':
            roles.append('special')
        elif tok == '[SEP]':
            roles.append('special')
            sep_count += 1
            in_doc = (sep_count >= 1)
        elif in_doc or seg == 1:
            roles.append('document')
        else:
            roles.append('query')

    # Print colour-coded
    markers = {'special': '\033[91m', 'query': '\033[94m', 'document': '\033[92m'}
    reset = '\033[0m'
    print('Cross-encoder tokenised input:')
    print()
    line = ''
    for tok, role in zip(tokens, roles):
        line += f'{markers[role]}{tok}{reset}  '
    print(line)
    print()
    print('\033[91mRed\033[0m = special tokens ([CLS], [SEP])')
    print('\033[94mBlue\033[0m = query tokens')
    print('\033[92mGreen\033[0m = document tokens')
    print()
    print('Unlike a bi-encoder, every blue (query) token can attend to')
    print('every green (document) token across all 6 Transformer layers.')
    print('This cross-attention is what makes relevance scoring so accurate.')

except Exception as e:
    print(f'Could not run tokeniser demo: {e}')
    print()
    print('Conceptual cross-encoder input (ms-marco-MiniLM-L-6-v2):')
    print()
    print('[CLS]  how  do  i  fix  oom  errors  ?  [SEP]  out  ##of  ##memory  ##error')
    print('       occurs  when  jvm  runs  out  of  memory  .  [SEP]')
    print()
    print('[CLS] = classification token — its final vector → relevance score')
    print('[SEP] = separator between query and document')

In [ ]:
# Step-by-step: what reranker.predict() actually does
print('What happens inside cross_encoder.predict(pairs):\n')
print('For each [query, document] pair:')
print('  1. Tokenise → [CLS] query [SEP] document [SEP]')
print('  2. Run through 6 Transformer layers (MiniLM) or 12 (BERT-base)')
print('     → every query token attends to every document token')
print('  3. Extract [CLS] token vector (1 × 384 dimensions)')
print('  4. Linear projection → 1 scalar relevance logit')
print()
print('Result for our OOM query:')
print()
print(f'{"Document":<40} {"Raw logit":<12} {"Interpretation"}')
print('-' * 72)
for idx in ce_order:
    score = ce_scores[idx]
    interp = 'highly relevant' if score > 5 else ('relevant' if score > 0 else 'not relevant')
    print(f'{labels[idx]:<40} {score:<12.3f} {interp}')

---
## 4. Why Can't We Use Cross-Encoders for Initial Retrieval?

Simple math:
- **Bi-encoder retrieval:** pre-computed doc vectors → O(1) lookup per doc → milliseconds for 1M docs  
- **Cross-encoder scoring:** full Transformer forward pass per pair → O(N) pairs → *hours* for 1M docs

In [ ]:
# Time a single cross-encoder call (warm up first)
_ = cross_encoder.predict([[query, docs[0]]])

N_TRIALS = 5
start = time.time()
for _ in range(N_TRIALS):
    cross_encoder.predict([[query, docs[0]]])
one_pair_ms = (time.time() - start) / N_TRIALS * 1000

print(f'One cross-encoder forward pass: {one_pair_ms:.1f} ms\n')
print('Scaling to full-corpus retrieval:')
print(f'{"Corpus size":<20} {"Total time":<20} {"Verdict"}')
print('-' * 55)

cases = [
    (1_000,     '1K'),
    (10_000,    '10K'),
    (100_000,   '100K'),
    (1_000_000, '1M'),
]
for n, label in cases:
    secs = one_pair_ms / 1000 * n
    if secs < 1:
        t_str = f'{secs*1000:.0f} ms'
    elif secs < 60:
        t_str = f'{secs:.1f} s'
    elif secs < 3600:
        t_str = f'{secs/60:.1f} min'
    else:
        t_str = f'{secs/3600:.1f} hrs'
    verdict = 'OK (small corpus)' if n <= 1_000 else ('Marginal' if n <= 10_000 else 'Way too slow')
    print(f'{label + " docs":<20} {t_str:<20} {verdict}')

print()
print('Solution: bi-encoder (vector DB) for Stage 1 — sub-linear with HNSW.')
print('Cross-encoder only for Stage 2 — linear in candidates, but candidates are small (20-50).')

---
## 5. Latency Analysis

The blog's claim: *"Reranking roughly doubles your retrieval latency — from ~50-100ms to ~150-400ms.  
But LLM generation dominates total latency anyway."*

Let's measure this directly.

In [ ]:
def measure_reranking_latency(n_docs, n_trials=3):
    # Build a candidate set of the target size by repeating our docs
    test_docs  = (docs * (n_docs // len(docs) + 1))[:n_docs]
    test_pairs = [[query, doc] for doc in test_docs]
    times = []
    for _ in range(n_trials):
        t0 = time.time()
        cross_encoder.predict(test_pairs)
        times.append((time.time() - t0) * 1000)
    return np.mean(times), np.std(times)

print('Measuring reranking latency (this takes ~30 seconds)...')
sizes = [5, 10, 20, 30, 50, 100]
latencies_ms = []
stds_ms = []
for n in sizes:
    mean_t, std_t = measure_reranking_latency(n)
    latencies_ms.append(mean_t)
    stds_ms.append(std_t)
    print(f'  {n:>4} candidates: {mean_t:>7.1f} ± {std_t:.1f} ms')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: absolute latency
axes[0].plot(sizes, latencies_ms, 'o-', color='#E53935', linewidth=2.5,
             markersize=7, label='Reranker latency', zorder=3)
axes[0].fill_between(sizes,
                     [m - s for m, s in zip(latencies_ms, stds_ms)],
                     [m + s for m, s in zip(latencies_ms, stds_ms)],
                     alpha=0.15, color='#E53935')
axes[0].axhspan( 10,  50, alpha=0.12, color='#42A5F5', label='Vector retrieval (10-50 ms)')
axes[0].axhspan(1000, 3000, alpha=0.08, color='#66BB6A', label='LLM generation (1000-3000 ms)')
for n, t in zip(sizes, latencies_ms):
    axes[0].annotate(f'{t:.0f}ms', (n, t), textcoords='offset points',
                     xytext=(4, 8), fontsize=8)
axes[0].set_xlabel('Candidates to Rerank')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Reranker Latency vs Candidate Set Size', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_xticks(sizes)

# Right: reranking as % of total pipeline
llm_ms = 1500
vector_ms = 30
totals = [vector_ms + t + llm_ms for t in latencies_ms]
pcts   = [t / total * 100 for t, total in zip(latencies_ms, totals)]
axes[1].bar(range(len(sizes)), pcts, color='#E53935', alpha=0.8)
axes[1].set_xticks(range(len(sizes)))
axes[1].set_xticklabels([str(n) for n in sizes])
axes[1].set_xlabel('Candidates to Rerank')
axes[1].set_ylabel('Reranking as % of total pipeline time')
axes[1].set_title('Reranking Overhead\n(assuming ~30ms retrieval + ~1500ms LLM)', fontweight='bold')
for i, pct in enumerate(pcts):
    axes[1].text(i, pct + 0.3, f'{pct:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print('Key insight: reranking adds 5-15% to total pipeline time,')
print('because LLM generation dominates at 1000-3000ms.')
print()
print('Sweet spot: 20-50 candidates — good coverage, predictable latency.')

---
## 6. The File Upload Example: Position 7 → Position 1

Reproducing the blog's before/after:

> **Query:** `What is the maximum file upload size in the Pro plan?`
>
> Hybrid search returns the correct answer document at **position 7** — it uses different
> terminology ("limits and quotas") and mentions "Pro" only once.  
> The reranker promotes it to **position 1** because it reads the full text and understands
> that "limits and quotas" is exactly what the user is asking about.

In [ ]:
from rank_bm25 import BM25Okapi

file_query = 'What is the maximum file upload size in the Pro plan?'

# Product documentation corpus — designed to mirror the blog's example
corpus = [
    {'id': 'doc_0', 'title': 'Pro Plan Overview',
     'text': 'The Pro plan includes advanced collaboration features, priority support, and enhanced storage. '
             'Pro users get access to our full API suite and can invite unlimited team members. '
             'The Pro plan is our most popular tier for growing teams.'},

    {'id': 'doc_1', 'title': 'File Upload API Documentation',
     'text': 'Our API supports file uploads for all plans. Enterprise plan customers can upload files up to 5GB. '
             'Use multipart upload for files larger than 100MB. See the developer guide for implementation details. '
             'The upload endpoint accepts standard HTTP multipart form-data requests.'},

    {'id': 'doc_2', 'title': 'Pro Plan Pricing Page',
     'text': 'The Pro plan is available at $49 per month billed annually. '
             'Pro plan includes 1TB storage, unlimited projects, and advanced analytics. '
             'Compare Pro with our Free and Enterprise tiers on the pricing page.'},

    {'id': 'doc_3', 'title': 'Getting Started with File Uploads',
     'text': 'Upload your first file in minutes. Drag and drop files into the dashboard, '
             'or use our API for programmatic uploads. Supported formats include PDF, DOCX, PNG, JPG, CSV.'},

    {'id': 'doc_4', 'title': 'API Rate Limits by Plan',
     'text': 'Free plan: 100 API requests per day. Pro plan: 10,000 API requests per day. '
             'Enterprise plan: unlimited. Rate limits reset at midnight UTC. '
             'Contact support if you need a temporary limit increase.'},

    {'id': 'doc_5', 'title': 'Storage and File Management',
     'text': 'Files are stored securely in encrypted cloud storage. '
             'Organise files with folders and tags. Auto-delete options available for temporary files. '
             'Storage usage is shown in your account dashboard.'},

    {'id': 'doc_6', 'title': 'Pro Plan Limits and Quotas',
     'text': 'Maximum file upload size: 100MB per file. Maximum total storage: 1TB. '
             'Maximum concurrent API connections: 50. Maximum team members: unlimited. '
             'These ceiling values apply per account. Contact support to discuss Enterprise options.'},

    {'id': 'doc_7', 'title': 'Billing and Subscription FAQ',
     'text': 'Pro plan is billed monthly or annually. Cancel anytime with no penalty. '
             'Includes a 14-day free trial. Switch between monthly and annual billing in account settings. '
             'Invoices are sent via email on your billing date.'},

    {'id': 'doc_8', 'title': 'Enterprise Plan Features',
     'text': 'Enterprise plan includes custom file upload limits up to 10GB per file, '
             'dedicated support, SSO integration, audit logs, and SLA guarantees. '
             'Contact our sales team for Enterprise pricing.'},

    {'id': 'doc_9', 'title': 'Troubleshooting Upload Errors',
     'text': 'If your upload fails: check the file size limit for your current plan, '
             'verify your network connection, confirm the file format is supported. '
             'Error code UPL-413 indicates the file exceeds the size limit for your plan.'},
]

TARGET_DOC_ID = 'doc_6'  # The correct answer — at position 7 in hybrid results
doc_id_to_idx  = {doc['id']: i for i, doc in enumerate(corpus)}

# ── Stage 1: Hybrid retrieval (BM25 + semantic + RRF) ─────────────────────────
tokenized = [doc['text'].lower().split() for doc in corpus]
bm25_index = BM25Okapi(tokenized)
bm25_raw   = bm25_index.get_scores(file_query.lower().split())
bm25_order = np.argsort(bm25_raw)[::-1]

q_emb = bi_encoder.encode(file_query, convert_to_tensor=True)
d_embs = bi_encoder.encode([doc['text'] for doc in corpus], convert_to_tensor=True)
sem_raw   = util.cos_sim(q_emb, d_embs)[0].numpy()
sem_order = np.argsort(sem_raw)[::-1]

def rrf_fuse(ranked_lists, k=60):
    scores = defaultdict(float)
    for rl in ranked_lists:
        for rank, idx in enumerate(rl, 1):
            scores[idx] += 1.0 / (k + rank)
    return sorted(scores.keys(), key=lambda x: scores[x], reverse=True)

hybrid_order = rrf_fuse([bm25_order.tolist(), sem_order.tolist()])

print(f'Query: {file_query!r}\n')
print('Hybrid search results (BEFORE reranking):')
print(f'{"Rank":<6} {"Doc":<8} {"Title":<35} {"BM25 pos":<10} {"Sem pos"}')
print('-' * 68)
for rank, idx in enumerate(hybrid_order[:8], 1):
    doc_id   = corpus[idx]['id']
    title    = corpus[idx]['title']
    b_pos    = list(bm25_order).index(idx) + 1
    s_pos    = list(sem_order).index(idx) + 1
    marker   = '  ← THE ANSWER' if doc_id == TARGET_DOC_ID else ''
    print(f'#{rank:<5} {doc_id:<8} {title:<35} #{b_pos:<9} #{s_pos}{marker}')

In [ ]:
# ── Stage 2: Cross-encoder reranking ──────────────────────────────────────────
N_CANDIDATES = 8
top_candidates = hybrid_order[:N_CANDIDATES]

rerank_pairs  = [[file_query, corpus[idx]['text']] for idx in top_candidates]
rerank_scores = cross_encoder.predict(rerank_pairs)

reranked = sorted(zip(rerank_scores, top_candidates), key=lambda x: x[0], reverse=True)

print('Results AFTER reranking:')
print(f'{"Rank":<6} {"Score":<8} {"Doc":<8} {"Title":<35} {"Hybrid rank"}')
print('-' * 72)
for rank, (score, idx) in enumerate(reranked, 1):
    doc_id     = corpus[idx]['id']
    title      = corpus[idx]['title']
    hybrid_pos = top_candidates.index(idx) + 1
    marker     = '  ← THE ANSWER' if doc_id == TARGET_DOC_ID else ''
    print(f'#{rank:<5} {score:<8.2f} {doc_id:<8} {title:<35} #{hybrid_pos}{marker}')

target_hybrid_pos = top_candidates.index(doc_id_to_idx[TARGET_DOC_ID]) + 1
target_rerank_pos = [i for i, (_, idx) in enumerate(reranked, 1)
                     if corpus[idx]['id'] == TARGET_DOC_ID][0]
print()
print(f'doc_6 (Pro Plan Limits) moved: hybrid rank #{target_hybrid_pos} → reranked #{target_rerank_pos}')
print()
print('Why hybrid search ranked it lower:')
print('  - Uses "limits and quotas" not "maximum size" → lower BM25 score')
print('  - Mentions "Pro" only once → cosine similarity diluted')
print('Why cross-encoder promoted it to #1:')
print('  - Reads full text + query together')
print('  - Understands "limits and quotas" = exactly what user is asking')

In [ ]:
# Before/after visualisation
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

def doc_color(idx):
    return '#2E7D32' if corpus[idx]['id'] == TARGET_DOC_ID else '#90CAF9'

titles_short = [corpus[i]['title'][:30] for i in top_candidates]

# Left: hybrid search (BM25 + semantic RRF scores)
hybrid_rrf_scores = []
for idx in top_candidates:
    b_r = list(bm25_order).index(idx) + 1
    s_r = list(sem_order).index(idx) + 1
    hybrid_rrf_scores.append(1/(60 + b_r) + 1/(60 + s_r))

y = np.arange(len(top_candidates))
before_colors = [doc_color(idx) for idx in top_candidates]
axes[0].barh(y, hybrid_rrf_scores, color=before_colors, alpha=0.85)
axes[0].set_yticks(y)
axes[0].set_yticklabels([f'#{r+1}: {corpus[top_candidates[r]]["title"][:30]}'
                          for r in range(len(top_candidates))], fontsize=9)
axes[0].set_xlabel('RRF Score')
axes[0].set_title('BEFORE Reranking\n(BM25 + Semantic + RRF)', fontweight='bold')

# Right: after reranking
ce_norm = rerank_scores - rerank_scores.min()
after_order = [idx for _, idx in reranked]
after_colors = [doc_color(idx) for idx in after_order]
after_scores = [ce_norm[top_candidates.index(idx)] for idx in after_order]
axes[1].barh(y, after_scores, color=after_colors, alpha=0.85)
axes[1].set_yticks(y)
axes[1].set_yticklabels([f'#{r+1}: {corpus[after_order[r]]["title"][:30]}'
                          for r in range(len(after_order))], fontsize=9)
axes[1].set_xlabel('Cross-Encoder Score (normalised)')
axes[1].set_title('AFTER Reranking\n(Cross-Encoder)', fontweight='bold')

legend_handles = [
    mpatches.Patch(color='#2E7D32', label=f'Correct answer ({TARGET_DOC_ID})'),
    mpatches.Patch(color='#90CAF9', label='Other documents'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2,
           fontsize=10, bbox_to_anchor=(0.5, -0.06))
fig.suptitle(f'Query: {file_query!r}', style='italic', fontsize=10, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Retrieval Diagnostic: Is Your Problem Stage 1 or Stage 2?

From the blog:

> *Take 20-30 queries where your RAG system gives suboptimal answers. If you frequently find
> that a better document exists at positions 5-15 but wasn't in the top-3 passed to the LLM,
> a reranker will help.*
>
> *If the better document isn't in your top-20 at all, your problem is in Stage 1 — fix
> chunking, embedding model, or hybrid search. A reranker won't help.*

Here's a diagnostic function that runs this check automatically.

In [ ]:
def diagnose_retrieval(query, ground_truth_doc_id, retriever_fn,
                       candidate_sizes=None, top_k_for_llm=3):
    """
    Diagnostic tool: check if the best document is in your candidate set,
    and whether a reranker is likely to help.

    Args:
        query:               the user query
        ground_truth_doc_id: doc ID of the known-best document
        retriever_fn:        fn(query, k) -> list of doc IDs (ranked)
        candidate_sizes:     list of k values to test
        top_k_for_llm:       how many docs the LLM receives
    """
    if candidate_sizes is None:
        candidate_sizes = [5, 10, 20, 50]

    print(f'Retrieval Diagnostic')
    print(f'  Query: {query!r}')
    print(f'  Looking for: {ground_truth_doc_id}')
    print(f'  LLM receives top-{top_k_for_llm}\n')
    print(f'{"Candidate k":<14} {"Found?":<10} {"At rank":<12} {"Reranker verdict"}')
    print('-' * 65)

    for k in candidate_sizes:
        candidate_ids = retriever_fn(query, k)
        found = ground_truth_doc_id in candidate_ids
        if found:
            pos = candidate_ids.index(ground_truth_doc_id) + 1
            if pos <= top_k_for_llm:
                verdict = 'Already in top-k — reranker not needed here'
            else:
                verdict = f'At #{pos}, outside top-{top_k_for_llm} — reranker WILL help'
        else:
            pos = None
            verdict = 'NOT FOUND — fix Stage 1 first, reranker cannot help'
        pos_str = f'#{pos}' if pos else 'absent'
        print(f'{k:<14} {str(found):<10} {pos_str:<12} {verdict}')


def hybrid_retriever(query, k):
    b_raw = bm25_index.get_scores(query.lower().split())
    q_emb = bi_encoder.encode(query, convert_to_tensor=True)
    s_raw = util.cos_sim(q_emb, d_embs)[0].numpy()
    order = rrf_fuse([np.argsort(b_raw)[::-1].tolist(),
                      np.argsort(s_raw)[::-1].tolist()])
    return [corpus[idx]['id'] for idx in order[:k]]


diagnose_retrieval(file_query, TARGET_DOC_ID, hybrid_retriever)

In [ ]:
# Test a query where Stage 1 completely misses — reranking won't save you
bad_query = 'ceiling value for uploading in premium subscription'
print('Testing a query with heavy vocabulary mismatch:\n')
diagnose_retrieval(bad_query, TARGET_DOC_ID, hybrid_retriever)
print()
print('If the doc is absent at k=50, the problem is in Stage 1 (recall).')
print('Options: improve chunking, use a better embedding model, add more hybrid signals.')
print()

# And a query where Stage 1 works well
good_query = 'Pro plan maximum upload size limit'
print('Testing a clear query:\n')
diagnose_retrieval(good_query, TARGET_DOC_ID, hybrid_retriever)

---
## 8. Complete RAG Retrieval Pipeline

Putting it all together: hybrid search → RRF → reranking → LLM.

This is the production-ready pattern from the blog.

In [ ]:
def rag_retrieve(query, top_k=3, n_candidates=20, verbose=True):
    """
    Complete two-stage RAG retrieval pipeline.

    Stage 1: Hybrid search — BM25 + dense embeddings + RRF fusion
    Stage 2: Cross-encoder reranking of top candidates

    Returns top_k document texts for the LLM.
    """
    timings = {}

    # ── Stage 1: fast hybrid retrieval ────────────────────────────────────────
    t0 = time.time()
    bm25_s = bm25_index.get_scores(query.lower().split())
    q_emb  = bi_encoder.encode(query, convert_to_tensor=True)
    sem_s  = util.cos_sim(q_emb, d_embs)[0].numpy()
    candidates = rrf_fuse([np.argsort(bm25_s)[::-1].tolist(),
                           np.argsort(sem_s)[::-1].tolist()])
    candidates = candidates[:n_candidates]  # cap candidate set
    timings['Stage 1 hybrid'] = (time.time() - t0) * 1000

    # ── Stage 2: precise reranking ─────────────────────────────────────────────
    t0 = time.time()
    pairs  = [[query, corpus[idx]['text']] for idx in candidates]
    scores = cross_encoder.predict(pairs)
    reranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    timings['Stage 2 rerank'] = (time.time() - t0) * 1000

    results = [corpus[idx] for _, idx in reranked[:top_k]]

    if verbose:
        print(f'Query: {query!r}\n')
        total_ms = sum(timings.values())
        for stage, ms in timings.items():
            print(f'  {stage}: {ms:.0f} ms')
        print(f'  LLM generation: ~1500 ms (not measured here)')
        print(f'  Retrieval total: {total_ms:.0f} ms')
        print()
        print(f'Top-{top_k} documents passed to the LLM:')
        for i, doc in enumerate(results, 1):
            snippet = doc['text'][:90] + '...'
            print(f'  #{i} [{doc["id"]}] {doc["title"]}')
            print(f'      {snippet}')

    return [doc['text'] for doc in results]


# Run the full pipeline
results = rag_retrieve(file_query, top_k=3, n_candidates=8)

---
## 9. Cohere Managed Reranking API

With a managed API, the cross-encoder runs on Cohere's infrastructure — no GPU needed.

Install: `pip install cohere`

In [ ]:
# ── Cohere reranking (requires cohere API key) ─────────────────────────────────
# Replace YOUR_API_KEY with your actual key from https://dashboard.cohere.com

COHERE_AVAILABLE = False
try:
    import cohere
    COHERE_AVAILABLE = True
except ImportError:
    pass

if COHERE_AVAILABLE:
    co = cohere.Client('YOUR_API_KEY')  # replace with your key

    retrieved_docs = [corpus[idx]['text'] for idx in hybrid_order[:10]]

    results = co.rerank(
        query=file_query,
        documents=retrieved_docs,
        top_n=3,
        model='rerank-english-v3.0',
    )

    print(f'Cohere rerank results for: {file_query!r}\n')
    for hit in results.results:
        snippet = hit.document.text[:80]
        print(f'  Score: {hit.relevance_score:.4f} | {snippet}...')
else:
    print('cohere not installed. Run: pip install cohere')
    print()
    print('Cohere reranking pattern:')
    cohere_code = '''
import cohere

co = cohere.Client("your-api-key")

results = co.rerank(
    query="What is the maximum file upload size in the Pro plan?",
    documents=retrieved_docs,   # list of strings from Stage 1
    top_n=3,
    model="rerank-english-v3.0",
)

for hit in results.results:
    print(f"Score: {hit.relevance_score:.4f} | {hit.document.text[:80]}...")
'''
    print(cohere_code)

---
## When to Add / Skip a Reranker — Quick Reference

### Add a reranker when:

| Signal | Explanation |
|---|---|
| Precision matters (legal, medical, support) | Wrong docs → hallucinations; reranker reduces this risk |
| You retrieve 20+, pass 3 to LLM | Wide gap between candidate set and final cut |
| Queries are complex / multi-faceted | CE checks every constraint; bi-encoder compresses them |
| Answers are "close but not quite right" | Best doc is in top-20 but not top-3 |
| Only doing dense retrieval (no hybrid) | CE adds cross-attention that cosine lacks |

### Skip the reranker when:

| Signal | Explanation |
|---|---|
| Sub-100ms latency SLA | Reranking adds 100-300ms |
| retrieve@1 is already accurate | Test: best doc at #1 for 90%+ of queries? |
| Candidate set ≤ 5 docs | LLM sees all anyway; reranking just adds latency |
| Very high query volume + cost constraints | Cross-encoder compute adds up at scale |

---

## Key Takeaways

1. **Bi-encoder vs cross-encoder is a speed/accuracy trade-off.** Bi-encoders pre-compute doc vectors (fast, shallow). Cross-encoders read query+doc together (slow, deep).

2. **The two-stage pattern is a principled engineering compromise.** Stage 1 maximises *recall*; Stage 2 maximises *precision*.

3. **The `[CLS] q [SEP] d [SEP]` format is the cross-encoder's secret weapon.** Full cross-attention means every query token can directly compare against every document token across all layers.

4. **Reranking adds ~100-300ms — small relative to LLM generation (1-3s).** With 20 candidates it's typically a 10-15% overhead on total pipeline time.

5. **The sweet spot is 20-50 candidates.** Fewer: you might miss good docs. More: diminishing returns with linear cost increase.

6. **Diagnose before adding a reranker.** If the best doc isn't in your top-20, fix Stage 1 first. A reranker cannot promote a document that retrieval never found.

---

*Up next: Lesson 6.5 — Evaluating your retrieval pipeline with nDCG, Recall@k, and MRR.*